# Eclat

## Importing the libraries

In [0]:
!pip install apyori

In [0]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd

## Data Preprocessing

In [0]:
dataset = pd.read_csv('Market_Basket_Optimisation.csv', header = None)
transactions = []
for i in range(0, 7501):
  transactions.append([str(dataset.values[i,j]) for j in range(0, 20)])

## Training the Eclat model on the dataset

In [0]:
from apyori import apriori
rules = apriori(transactions = transactions, min_support = 0.003, min_confidence = 0.2, min_lift = 3, min_length = 2, max_length = 2)

## Visualising the results

### Displaying the first results coming directly from the output of the apriori function

In [0]:
results = list(rules)

In [0]:
results

[RelationRecord(items=frozenset({'light cream', 'chicken'}), support=0.004532728969470737, ordered_statistics=[OrderedStatistic(items_base=frozenset({'light cream'}), items_add=frozenset({'chicken'}), confidence=0.29059829059829057, lift=4.84395061728395)]),
 RelationRecord(items=frozenset({'escalope', 'mushroom cream sauce'}), support=0.005732568990801226, ordered_statistics=[OrderedStatistic(items_base=frozenset({'mushroom cream sauce'}), items_add=frozenset({'escalope'}), confidence=0.3006993006993007, lift=3.790832696715049)]),
 RelationRecord(items=frozenset({'escalope', 'pasta'}), support=0.005865884548726837, ordered_statistics=[OrderedStatistic(items_base=frozenset({'pasta'}), items_add=frozenset({'escalope'}), confidence=0.3728813559322034, lift=4.700811850163794)]),
 RelationRecord(items=frozenset({'honey', 'fromage blanc'}), support=0.003332888948140248, ordered_statistics=[OrderedStatistic(items_base=frozenset({'fromage blanc'}), items_add=frozenset({'honey'}), confidence=0

### Putting the results well organised into a Pandas DataFrame

In [0]:
def inspect(results):
    lhs         = [tuple(result[2][0][0])[0] for result in results]
    rhs         = [tuple(result[2][0][1])[0] for result in results]
    supports    = [result[1] for result in results]
    return list(zip(lhs, rhs, supports))
resultsinDataFrame = pd.DataFrame(inspect(results), columns = ['Product 1', 'Product 2', 'Support'])

### Displaying the results sorted by descending supports

In [0]:
resultsinDataFrame.nlargest(n = 10, columns = 'Support')

,Product 1,Product 2,Support
4,herb & pepper,ground beef,0.015998
7,whole wheat pasta,olive oil,0.007999
2,pasta,escalope,0.005866
1,mushroom cream sauce,escalope,0.005733
5,tomato sauce,ground beef,0.005333
8,pasta,shrimp,0.005066
0,light cream,chicken,0.004533
3,fromage blanc,honey,0.003333
6,light cream,olive oil,0.003200


# Practical Eclat Study Notes

## Objective

This exercise analyzes product combinations in the Market Basket Optimization dataset. Each row represents a transaction, and the goal is to rank two-product itemsets by support:

$$\operatorname{support}(X)=\frac{\text{transactions containing every item in }X}{\text{total transactions}}$$

For the proposed `buy one product, get another product free` campaign, the notebook focuses on pairs such as `{herb & pepper, ground beef}`. Because an itemset is unordered, the labels **Product 1** and **Product 2** are more appropriate than **Left Hand Side** and **Right Hand Side** when discussing support alone.

## Workflow used in the notebook

The practical implementation follows these steps:

1. Load the 7,501 market-basket transactions.
2. Convert every basket into a list of product strings.
3. Call `apyori.apriori()` with minimum thresholds and a maximum itemset length of two.
4. Convert the returned generator to a list.
5. Extract two product labels and the support value from each result.
6. Build a pandas DataFrame.
7. Retrieve the highest-support rows with:

```python
resultsinDataFrame.nlargest(n=10, columns='Support')
```

`nlargest()` orders the selected rows by support from highest to lowest. This notebook has only nine results that pass all its filters, so requesting ten rows returns nine.

## Interpreting the displayed results

| Product 1 | Product 2 | Support | Approximate percentage |
|---|---|---:|---:|
| herb & pepper | ground beef | 0.015998 | 1.60% |
| whole wheat pasta | olive oil | 0.007999 | 0.80% |
| pasta | escalope | 0.005866 | 0.59% |
| mushroom cream sauce | escalope | 0.005733 | 0.57% |
| tomato sauce | ground beef | 0.005333 | 0.53% |
| pasta | shrimp | 0.005066 | 0.51% |
| light cream | chicken | 0.004533 | 0.45% |
| fromage blanc | honey | 0.003333 | 0.33% |
| light cream | olive oil | 0.003200 | 0.32% |

The most frequent displayed pair is `{herb & pepper, ground beef}`, occurring in approximately 1.6% of all transactions.

## Important implementation distinction

> **This notebook does not execute the Eclat algorithm.** It executes the `apyori` package's Apriori implementation and presents part of its output in an Eclat-style, support-focused table.

A genuine Eclat implementation organizes the database vertically as item-to-transaction-ID sets and mines frequent itemsets through transaction-ID intersections and a bottom-up lattice traversal. Changing column names or hiding confidence and lift does not change Apriori into Eclat.

The notebook is still useful for learning how to rank itemsets by support, but it should be described as an **Apriori-based support analysis** or an **Eclat-style output**, not as a true Eclat implementation.

## Why the current output is not support-only

The training cell uses:

```python
rules = apriori(
    transactions=transactions,
    min_support=0.003,
    min_confidence=0.2,
    min_lift=3,
    min_length=2,
    max_length=2
)
```

Although the final DataFrame does not display confidence or lift, `min_confidence=0.2` and `min_lift=3` still filter the results internally. Consequently, the table does **not** contain every pair meeting the support threshold. It contains only itemsets that also produce at least one directional rule meeting the confidence and lift thresholds.

Keeping those filters may produce a useful shortlist, but it is not a pure support analysis.

## A support-only Apriori proxy

If the goal is to use `apyori` to obtain support-focused pair itemsets, remove the directional filtering and read each `RelationRecord.items` collection directly:

```python
from apyori import apriori

records = list(apriori(
    transactions=transactions,
    min_support=0.003,
    min_confidence=0,
    min_lift=0,
    max_length=2
))

pair_rows = []
for record in records:
    items = sorted(record.items)
    if len(items) == 2:
        pair_rows.append((items[0], items[1], record.support))

resultsinDataFrame = pd.DataFrame(
    pair_rows,
    columns=['Product 1', 'Product 2', 'Support']
)

resultsinDataFrame.nlargest(10, 'Support')
```

This remains Apriori rather than Eclat, but its output now reflects support filtering rather than confidence-and-lift filtering.

### `min_length` caveat

In `apyori` 1.1.2, `apriori()` reads `min_support`, `min_confidence`, `min_lift`, and `max_length`; it does not implement `min_length`. The supplied `min_length=2` argument is silently ignored. Filter the returned records explicitly with `len(record.items) == 2` when exact pair length is required.

## Itemsets larger than two products

Increasing `max_length` alone is not sufficient with the notebook's current `inspect()` function. That function extracts the first directional `ordered_statistics` entry and assumes exactly one item on each side. It can omit items or misrepresent larger itemsets.

For variable-length itemsets, store the complete item collection in one column:

```python
itemset_rows = [
    (tuple(sorted(record.items)), record.support)
    for record in records
    if len(record.items) >= 2
]

itemsets_df = pd.DataFrame(itemset_rows, columns=['Itemset', 'Support'])
itemsets_df.nlargest(10, 'Support')
```

An itemset such as `{A, B, C}` has one support value regardless of how it might later be divided into a directional rule.

## Data-preparation caution

The current preprocessing converts all 20 cells in every row to strings. Empty CSV cells therefore become the artificial item `'nan'`. A safer transaction builder excludes missing values:

```python
transactions = []
for i in range(len(dataset)):
    transaction = [
        str(value)
        for value in dataset.iloc[i]
        if pd.notna(value)
    ]
    transactions.append(transaction)
```

## Choosing between Apriori and Eclat

Neither algorithm is universally best:

- **Apriori** commonly performs level-wise candidate generation and is intuitive to study.
- **Eclat** uses vertical transaction-ID sets and typically explores itemsets depth-first.
- Eclat can be efficient when transaction-ID intersections are compact and fast.
- Large transaction-ID sets can consume substantial memory.
- The preferred algorithm depends on transaction count, basket density, support threshold, memory, implementation quality, and desired output.

Confidence and lift are not exclusive advantages of Apriori. They are rule-evaluation metrics that can also be calculated after Eclat has mined frequent itemsets.

## Business interpretation

High support identifies common combinations, but it does not prove that a `buy one, get one free` offer will be profitable. Before launching a promotion, consider product margins, baseline demand, stock constraints, cannibalization, customer segments, and controlled experimental results.

## Quick review

1. **What does the current notebook actually run?** The `apyori` implementation of Apriori.
2. **Why are the displayed results not support-only?** Confidence and lift still filter records before the DataFrame is created.
3. **What does `nlargest(10, 'Support')` do?** It returns up to ten rows with the greatest support values in descending order.
4. **Does `min_length=2` work in `apyori` 1.1.2?** No; exact itemset length must be filtered explicitly.
5. **How should larger itemsets be stored?** As complete collections such as tuples, rather than forced into two single-product columns.
6. **What distinguishes genuine Eclat?** Vertical transaction-ID sets, intersections, and Eclat's lattice-search procedure.
7. **Can rules be generated after Eclat?** Yes. Confidence and lift can be calculated after frequent itemsets have been mined.

## Key takeaway

The notebook demonstrates how to rank product combinations by support using Apriori-generated results. Treat it as a support-analysis exercise. For a genuine Eclat implementation, use an algorithm that explicitly mines vertical transaction-ID sets, and for a genuinely support-only output, remove confidence and lift filtering and extract complete itemsets directly.
